In [1]:
# ============================================================
# 02 - Prétraitement des données NSL-KDD
# ============================================================

import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, MinMaxScaler

# Noms des colonnes
columns = [
    "duration", "protocol_type", "service", "flag", "src_bytes",
    "dst_bytes", "land", "wrong_fragment", "urgent", "hot",
    "num_failed_logins", "logged_in", "num_compromised", "root_shell",
    "su_attempted", "num_root", "num_file_creations", "num_shells",
    "num_access_files", "num_outbound_cmds", "is_host_login",
    "is_guest_login", "count", "srv_count", "serror_rate",
    "srv_serror_rate", "rerror_rate", "srv_rerror_rate", "same_srv_rate",
    "diff_srv_rate", "srv_diff_host_rate", "dst_host_count",
    "dst_host_srv_count", "dst_host_same_srv_rate", "dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate", "dst_host_srv_diff_host_rate",
    "dst_host_serror_rate", "dst_host_srv_serror_rate", "dst_host_rerror_rate",
    "dst_host_srv_rerror_rate", "label", "difficulty"
]

# Chargement
train = pd.read_csv("../data/raw/KDDTrain+.txt", names=columns)
test  = pd.read_csv("../data/raw/KDDTest+.txt",  names=columns)

# Supprimer la colonne difficulty (non utile)
train.drop("difficulty", axis=1, inplace=True)
test.drop("difficulty",  axis=1, inplace=True)

print("✅ Données chargées !")
print(f"Train : {train.shape} | Test : {test.shape}")

✅ Données chargées !
Train : (125973, 42) | Test : (22544, 42)


In [2]:
# ============================================================
# Étape 1 - Encoder la colonne cible (label)
# Normal = 0 / Attaque = 1
# ============================================================

train["label"] = train["label"].apply(lambda x: 0 if x == "normal" else 1)
test["label"]  = test["label"].apply(lambda x: 0 if x == "normal" else 1)

print("=== Répartition Train ===")
print(train["label"].value_counts())
print("\n=== Répartition Test ===")
print(test["label"].value_counts())

=== Répartition Train ===
label
0    67343
1    58630
Name: count, dtype: int64

=== Répartition Test ===
label
1    12833
0     9711
Name: count, dtype: int64


In [3]:
# ============================================================
# Étape 2 - Encoder les colonnes texte
# protocol_type, service, flag → valeurs numériques
# ============================================================

le = LabelEncoder()

colonnes_texte = ["protocol_type", "service", "flag"]

for col in colonnes_texte:
    train[col] = le.fit_transform(train[col])
    test[col]  = le.transform(test[col])

print("✅ Colonnes texte encodées !")
print(train[["protocol_type", "service", "flag"]].head())

✅ Colonnes texte encodées !
   protocol_type  service  flag
0              1       20     9
1              2       44     9
2              1       49     5
3              1       24     9
4              1       24     9


In [4]:
# ============================================================
# Étape 3 - Normalisation des données (mise à l'échelle 0-1)
# ============================================================

# Séparer features et cible
X_train = train.drop("label", axis=1)
y_train = train["label"]

X_test = test.drop("label", axis=1)
y_test = test["label"]

# Normalisation
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print("✅ Normalisation terminée !")
print(f"X_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}")
print(f"y_train : {y_train.shape}")
print(f"y_test  : {y_test.shape}")

✅ Normalisation terminée !
X_train : (125973, 41)
X_test  : (22544, 41)
y_train : (125973,)
y_test  : (22544,)


In [5]:
# ============================================================
# Étape 4 - Sauvegarder les données prétraitées
# ============================================================

import os

# Convertir en DataFrame pour sauvegarder
X_train_df = pd.DataFrame(X_train)
X_test_df  = pd.DataFrame(X_test)

# Sauvegarder dans data/processed/
X_train_df.to_csv("../data/processed/X_train.csv", index=False)
X_test_df.to_csv("../data/processed/X_test.csv",   index=False)
y_train.to_csv("../data/processed/y_train.csv",     index=False)
y_test.to_csv("../data/processed/y_test.csv",       index=False)

print("✅ Données prétraitées sauvegardées dans data/processed/")
print(os.listdir("../data/processed/"))

✅ Données prétraitées sauvegardées dans data/processed/
['X_test.csv', 'X_train.csv', 'y_test.csv', 'y_train.csv']
